# ARC-AGI Dataset Consolidation and Single-File Export

This notebook automates the process of fetching the official Abstraction and Reasoning Corpus (ARC) dataset, parsing the dispersed task-specific JSON files, and consolidating them into two single-file aggregates: one for **training data** and one for **evaluation data**. This consolidated format simplifies importing tasks into downstream downstream applications, machine learning pipelines, and related web interfaces.

## 1. Environment Setup and Google Drive Mounting

To ensure seamless execution in Google Colab and local environments, we mount Google Drive and establish path structures for exporting the consolidated files.

In [1]:
import os
import sys

# Detect if running in Google Colab and handle mounting
try:
    from google.colab import drive
    drive.mount('/content/drive')
    export_dir = '/content/drive/MyDrive/motifs/'
    os.makedirs(export_dir, exist_ok=True)
    print('Google Drive mounted successfully. Export directory set to:', export_dir)
except ImportError:
    export_dir = './motifs/'
    os.makedirs(export_dir, exist_ok=True)
    print('Running locally. Export directory set to:', export_dir)


Running locally. Export directory set to: ./motifs/


## 2. Fetching ARC-AGI Dataset

We fetch the entire ARC-AGI repository as a ZIP archive directly from GitHub. This approach is more efficient and reliable than sending hundreds of individual HTTP requests for separate task files.

In [2]:
import urllib.request
import zipfile

zip_url = 'https://github.com/fchollet/ARC/archive/refs/heads/master.zip'
zip_path = 'arc_master.zip'

print(f'Downloading ARC dataset zip from {zip_url}...')
urllib.request.urlretrieve(zip_url, zip_path)
print('Download completed. Extracting archive...')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('arc_extracted')
print('Extraction complete. Temporary files stored in "arc_extracted".')


Download completed. Extracting archive...


Extraction complete. Temporary files stored in "arc_extracted".


## 3. Parsing and Consolidating Tasks

We iterate through the extracted files for the training and evaluation splits, loading each JSON task, mapping its file name (Task ID) as the dictionary key, and building two consolidated dictionaries.

In [3]:
import json
import glob

def consolidate_split(split_name):
    # Find all task JSON files for the given split inside the extracted archive
    search_path = os.path.join('arc_extracted', 'ARC-AGI-master', 'data', split_name, '*.json')
    task_files = glob.glob(search_path)
    print(f'Found {len(task_files)} task files for split: {split_name}')

    consolidated = {}
    for file_path in sorted(task_files):
        task_id = os.path.splitext(os.path.basename(file_path))[0]
        with open(file_path, 'r') as f:
            consolidated[task_id] = json.load(f)
    return consolidated

training_data = consolidate_split('training')
evaluation_data = consolidate_split('evaluation')


Found 400 task files for split: training
Found 400 task files for split: evaluation


## 4. Exporting to Single-File JSON

We save the consolidated dictionaries to single-file JSONs under the standard export path directory, supporting both local and Google Drive layouts.

In [4]:
train_export_path = os.path.join(export_dir, 'arc_training_consolidated.json')
eval_export_path = os.path.join(export_dir, 'arc_evaluation_consolidated.json')

with open(train_export_path, 'w') as f:
    json.dump(training_data, f, indent=2)
print(f'Successfully exported training data to: {train_export_path}')

with open(eval_export_path, 'w') as f:
    json.dump(evaluation_data, f, indent=2)
print(f'Successfully exported evaluation data to: {eval_export_path}')


Successfully exported training data to: ./motifs/arc_training_consolidated.json


Successfully exported evaluation data to: ./motifs/arc_evaluation_consolidated.json


## 5. Verification and Cleanup

We verify that the generated files exist, contain valid JSON objects, and clean up the downloaded ZIP archive and intermediate extraction directories to keep our working tree clean.

In [5]:
import shutil

# Verify exported sizes
with open(train_export_path, 'r') as f:
    train_verify = json.load(f)
with open(eval_export_path, 'r') as f:
    eval_verify = json.load(f)

print('Verification successful!')
print(f'Consolidated training file has {len(train_verify)} tasks.')
print(f'Consolidated evaluation file has {len(eval_verify)} tasks.')

# Cleanup temporary files
if os.path.exists('arc_master.zip'):
    os.remove('arc_master.zip')
if os.path.exists('arc_extracted'):
    shutil.rmtree('arc_extracted')
print('Temporary extraction assets removed successfully.')


Verification successful!
Consolidated training file has 400 tasks.
Consolidated evaluation file has 400 tasks.
Temporary extraction assets removed successfully.
